In [ ]:
pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.3 MB/s eta 0:00:00


In [ ]:

from pathlib import Path
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from reportlab.lib import colors
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, KeepTogether

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
np.random.seed(20260404)

# ---------------------------------------------------------------------
# Output files
# ---------------------------------------------------------------------

FIG1_PDF = "Figure_1_model_diagram_markov_structure_service_pathway.pdf"
TABLE1_PDF = "Table_1_input_parameters_and_PSA_distributions.pdf"
TABLE1_CSV = "Table_1_input_parameters_and_PSA_distributions.csv"

# ---------------------------------------------------------------------
# Figure 1 content
# ---------------------------------------------------------------------
FIGURE_TITLE = "Figure 1. Model diagram: Markov structure and malaria service pathway"
FIGURE_CAPTION = (
    "Panel A shows the monthly open-cohort Markov model for children under 5 years of age. "
    "Children enter through births, move among the Healthy, Uncomplicated malaria, Severe malaria, "
    "and Death states, and exit the modeled population when they reach 5 years of age. Death is "
    "absorbing. Panel B shows the malaria service pathway nested within uncomplicated malaria. "
    "Episodes follow the treated transition pathway only when rapid diagnostic tests (RDTs) are in "
    "stock, the child is tested, and prompt artemisinin-based combination therapy (ACT) is given "
    "after a positive test. Costs are modeled separately and are not shown."
)

# ---------------------------------------------------------------------
# Table 1 content
# ---------------------------------------------------------------------
rows = [
    # domain, parameter, base value, distribution, notes
    ("Demography", "Facility catchment population", "9,000", "Fixed", "Per-facility population served"),
    ("Demography", "Children under 5 years", "18.0%", "Fixed", "Study population share"),
    ("Demography", "Life expectancy at age 0", "62 years", "Truncated Normal(62, 8); bounds 45–80", "Varied in PSA"),
    ("Service pathway", "RDT in stock, control arm", "0.81", "Fixed", "Held fixed in PSA"),
    ("Service pathway", "Increment in RDT availability", "0.15", "Truncated Normal(0.15, 0.04); bounds 0.05–0.25", "Trial-informed intervention effect"),
    ("Service pathway", "Testing given stock, control arm", "0.60", "Fixed", "Held fixed in PSA"),
    ("Service pathway", "Increment in testing given stock", "0.19", "Truncated Normal(0.19, 0.05); bounds 0.05–0.33", "Trial-informed intervention effect"),
    ("Service pathway", "Prompt ACT after positive test, control arm", "0.82", "Fixed", "Held fixed in PSA"),
    ("Service pathway", "Pass-through from added testing to prompt ACT", "0.82", "Fixed", "Structural sensitivity outside PSA"),
    ("Clinical transition", "Healthy → Death (annual)", "0.007", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Healthy → Uncomplicated malaria (annual)", "0.400", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Uncomplicated malaria → Healthy (annual)", "0.861", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Uncomplicated malaria → Severe malaria (annual)", "0.130", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Uncomplicated malaria → Death (annual)", "0.009", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Severe malaria → Healthy (annual)", "0.938", "Beta(mean, k=100)", "Annual input probability"),
    ("Clinical transition", "Severe malaria → Death (annual)", "0.062", "Beta(mean, k=100)", "Annual input probability"),
    ("Disability weight", "Uncomplicated malaria", "0.006", "Beta(mean, k=100)", "Varied in PSA"),
    ("Disability weight", "Severe malaria", "0.133", "Beta(mean, k=100)", "Varied in PSA"),
    ("Costs", "Medical-cost multiplier", "1.00", "Gamma(mean=1, CV=0.25)", "Applies to malaria-related medical costs only"),
    ("Costs", "Program-cost uncertainty", "Three structural scenarios", "Not separately randomized", "Handled through Scenarios 1–3"),
]

TABLE_TITLE = "Table 1. Input parameters and PSA distributions"
TABLE_NOTE = (
    "Note: Annual clinical transition probabilities are converted to monthly transition probabilities "
    "inside the model using a competing-risks formulation. Prompt ACT pass-through is treated as a "
    "structural assumption and is examined in deterministic sensitivity analysis rather than randomized "
    "in the probabilistic sensitivity analysis (PSA). Program cost uncertainty is represented through "
    "the three deterministic pricing and scale-up scenarios rather than by a separate stochastic multiplier."
)

# ---------------------------------------------------------------------
# Helpers for Figure 1
# ---------------------------------------------------------------------
def add_box(ax, x, y, w, h, text, fc="#F8F9FB", ec="#3B4559", fontsize=11, weight="normal"):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.4,
        edgecolor=ec,
        facecolor=fc,
    )
    ax.add_patch(patch)
    ax.text(
        x + w / 2, y + h / 2, text,
        ha="center", va="center", fontsize=fontsize, weight=weight, linespacing=1.15
    )
    return patch


def add_arrow(ax, start, end, text=None, text_offset=(0, 0), color="#3B4559", lw=1.4, style="-|>", rad=0.0):
    arrow = FancyArrowPatch(
        start, end,
        arrowstyle=style,
        mutation_scale=12,
        linewidth=lw,
        color=color,
        connectionstyle=f"arc3,rad={rad}",
        shrinkA=4, shrinkB=4,
    )
    ax.add_patch(arrow)
    if text:
        mx = (start[0] + end[0]) / 2 + text_offset[0]
        my = (start[1] + end[1]) / 2 + text_offset[1]
        ax.text(mx, my, text, fontsize=9.5, ha="center", va="center")
    return arrow


def build_markov_panel(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.02, 0.96, "A. Open-cohort Markov model", fontsize=13, weight="bold", ha="left")

    add_box(ax, 0.02, 0.64, 0.20, 0.15, "Children under 5\n\nBirths enter here\nmonthly", fc="#EEF3FA", fontsize=11, weight="bold")
    ax.text(0.12, 0.58, "Age out at 5 years", fontsize=9.5, ha="center")

    add_box(ax, 0.30, 0.64, 0.17, 0.15, "Healthy", fc="#F4F6F8", fontsize=12, weight="bold")
    ax.text(0.385, 0.58, "remain healthy", fontsize=9.5, ha="center")

    add_box(ax, 0.56, 0.64, 0.20, 0.15, "Uncomplicated\nmalaria", fc="#EAF4FE", fontsize=12, weight="bold")
    ax.text(0.66, 0.58, "service pathway\nshown in Panel B", fontsize=9.5, ha="center")

    add_box(ax, 0.56, 0.26, 0.20, 0.15, "Severe\nmalaria", fc="#FCEFEA", fontsize=12, weight="bold")
    ax.text(0.66, 0.20, "remain severe", fontsize=9.5, ha="center")

    add_box(ax, 0.83, 0.45, 0.14, 0.15, "Death", fc="#F5F1F1", fontsize=12, weight="bold")
    ax.text(0.90, 0.39, "absorbing state", fontsize=9.5, ha="center")

    add_arrow(ax, (0.22, 0.715), (0.30, 0.715))
    add_arrow(ax, (0.47, 0.715), (0.56, 0.715))
    add_arrow(ax, (0.47, 0.69), (0.83, 0.53), rad=-0.05)

    add_arrow(ax, (0.66, 0.64), (0.66, 0.41))
    add_arrow(ax, (0.56, 0.33), (0.47, 0.71), text="recovery", text_offset=(-0.06, 0.00), rad=0.15)
    add_arrow(ax, (0.76, 0.34), (0.83, 0.50))


def build_service_panel(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.02, 0.96, "B. Service pathway within uncomplicated malaria", fontsize=13, weight="bold", ha="left")

    add_box(ax, 0.04, 0.77, 0.22, 0.13, "Incident uncomplicated\nmalaria episode", fc="#EAF4FE", fontsize=11, weight="bold")
    add_box(ax, 0.33, 0.77, 0.18, 0.13, "RDT in stock?", fc="#FFF8E8", fontsize=11, weight="bold")
    add_box(ax, 0.58, 0.77, 0.16, 0.13, "Child tested?", fc="#FFF8E8", fontsize=11, weight="bold")
    add_box(ax, 0.80, 0.77, 0.17, 0.13, "Prompt ACT\nafter positive\ntest?", fc="#FFF8E8", fontsize=10.5, weight="bold")

    add_box(ax, 0.40, 0.46, 0.24, 0.15, "Untreated\nclinical pathway", fc="#F7F7F7", fontsize=11, weight="bold")
    add_box(ax, 0.77, 0.46, 0.19, 0.15, "Treated\nclinical pathway", fc="#E8F6EF", fontsize=11, weight="bold")

    add_box(ax, 0.27, 0.12, 0.20, 0.15, "Recover to\nhealthy", fc="#F4F6F8", fontsize=11, weight="bold")
    add_box(ax, 0.52, 0.12, 0.20, 0.15, "Severe\nmalaria", fc="#FCEFEA", fontsize=11, weight="bold")
    add_box(ax, 0.77, 0.12, 0.16, 0.15, "Death", fc="#F5F1F1", fontsize=11, weight="bold")

    add_arrow(ax, (0.26, 0.835), (0.33, 0.835))
    add_arrow(ax, (0.51, 0.835), (0.58, 0.835), text="Yes", text_offset=(0, 0.035))
    add_arrow(ax, (0.74, 0.835), (0.80, 0.835), text="Yes", text_offset=(0, 0.035))

    add_arrow(ax, (0.42, 0.77), (0.50, 0.61), text="No", text_offset=(-0.03, 0.025), rad=-0.05)
    add_arrow(ax, (0.66, 0.77), (0.53, 0.61), text="No", text_offset=(-0.05, 0.015), rad=0.05)
    add_arrow(ax, (0.88, 0.77), (0.60, 0.61), text="No", text_offset=(-0.07, 0.015), rad=0.10)
    add_arrow(ax, (0.88, 0.77), (0.86, 0.61), text="Yes", text_offset=(0.04, 0.02))

    ax.text(0.52, 0.41, "reference transition risks", fontsize=9.0, ha="center")
    ax.text(0.865, 0.41, "higher recovery,\nlower death", fontsize=9.0, ha="center")

    add_arrow(ax, (0.46, 0.46), (0.37, 0.27))
    add_arrow(ax, (0.52, 0.46), (0.62, 0.27))
    add_arrow(ax, (0.58, 0.46), (0.85, 0.27))

    add_arrow(ax, (0.81, 0.46), (0.43, 0.27), rad=0.10)
    add_arrow(ax, (0.86, 0.46), (0.62, 0.27))
    add_arrow(ax, (0.91, 0.46), (0.85, 0.27))


def create_figure_1():
    fig = plt.figure(figsize=(15, 9))
    gs = fig.add_gridspec(1, 2, left=0.04, right=0.98, top=0.90, bottom=0.17, wspace=0.08)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    fig.text(0.04, 0.96, FIGURE_TITLE, fontsize=16, weight="bold", ha="left")
    build_markov_panel(ax1)
    build_service_panel(ax2)

    wrapped = textwrap.fill(FIGURE_CAPTION, width=165)
    fig.text(0.04, 0.055, wrapped, fontsize=10.2, ha="left", va="bottom")
    fig.savefig(FIG1_PDF, format="pdf")
    plt.close(fig)


# ---------------------------------------------------------------------
# Table 1 creation
# ---------------------------------------------------------------------
def create_table_1():
    import pandas as pd

    df = pd.DataFrame(rows, columns=[
        "Domain", "Parameter", "Base-case value", "Distribution in PSA", "Notes"
    ])
    df.to_csv(TABLE1_CSV, index=False)

    doc = SimpleDocTemplate(
        str(TABLE1_PDF),
        pagesize=landscape(letter),
        leftMargin=0.45 * inch,
        rightMargin=0.45 * inch,
        topMargin=0.45 * inch,
        bottomMargin=0.45 * inch,
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "TitleStyle",
        parent=styles["Heading1"],
        fontName="Helvetica-Bold",
        fontSize=15,
        leading=18,
        alignment=TA_LEFT,
        textColor=colors.HexColor("#1F2A44"),
        spaceAfter=8,
    )
    note_style = ParagraphStyle(
        "NoteStyle",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=8.2,
        leading=10,
        alignment=TA_LEFT,
        textColor=colors.HexColor("#222222"),
        spaceBefore=6,
    )
    cell_style = ParagraphStyle(
        "CellStyle",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=8.0,
        leading=9.5,
        alignment=TA_LEFT,
        textColor=colors.black,
    )
    header_style = ParagraphStyle(
        "HeaderStyle",
        parent=styles["BodyText"],
        fontName="Helvetica-Bold",
        fontSize=8.4,
        leading=9.5,
        alignment=TA_CENTER,
        textColor=colors.white,
    )

    def p(text, style):
        return Paragraph(str(text).replace("\n", "<br/>"), style)

    data = [[
        p("Domain", header_style),
        p("Parameter", header_style),
        p("Base-case value", header_style),
        p("Distribution in PSA", header_style),
        p("Notes", header_style),
    ]]
    for r in rows:
        data.append([p(v, cell_style) for v in r])

    col_widths = [1.45 * inch, 2.65 * inch, 1.10 * inch, 2.45 * inch, 2.75 * inch]
    table = Table(data, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#365A8C")),
        ("LINEBELOW", (0, 0), (-1, 0), 1.0, colors.HexColor("#27466D")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7F9FC")]),
        ("BOX", (0, 0), (-1, -1), 0.8, colors.HexColor("#9AA7B8")),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#C8D1DD")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))

    story = [
        Paragraph(TABLE_TITLE, title_style),
        table,
        Spacer(1, 0.12 * inch),
        Paragraph(TABLE_NOTE, note_style),
    ]
    doc.build(story)


def main():
    create_figure_1()
    create_table_1()


if __name__ == "__main__":
    main()
